In [31]:
pip install langchain langchain-google-genai google-generativeai langchain_community faiss-cpu

   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ------ --------------------------------- 2.4/13.7 MB 11.2 MB/s eta 0:00:02
   -------------- ------------------------- 5.0/13.7 MB 11.6 MB/s eta 0:00:01
   ---------------------- ----------------- 7.6/13.7 MB 11.7 MB/s eta 0:00:01
   ----------------------------- ---------- 10.0/13.7 MB 11.7 MB/s eta 0:00:01
   ------------------------------------ --- 12.3/13.7 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------  13.6/13.7 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 13.7/13.7 MB 10.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [25]:

specs = {}
specs['GOOGLE_API_KEY'] = ''
specs['MODEL_NAME'] = "gemini-1.5-pro-001"
specs['TOKENS'] = 30
specs['EMBED_MODEL_NAME'] = "models/text-embedding-004"


In [12]:
from langchain_google_genai import GoogleGenerativeAI

llm = GoogleGenerativeAI(model=specs['MODEL_NAME'],max_tokens=specs['TOKENS'],api_key=specs['GOOGLE_API_KEY'])




In [11]:
# llm.invoke("Explain a Large Language Model in one line?")
llm.invoke("What is national sport of canada")

AIMessage(content='The national sports of Canada are **lacrosse** (summer sport) and **ice hockey** (winter sport). \n\n* **Lacrosse**', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-1.5-pro-001', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-9ea143b3-ba29-472e-810e-40c08f4fe506-0', usage_metadata={'input_tokens': 7, 'output_tokens': 30, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}})

In [10]:
from langchain.schema import SystemMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=specs['MODEL_NAME'],api_key=specs['GOOGLE_API_KEY'],max_tokens=specs['TOKENS'])

llm([
    SystemMessage(content='You should act as an experienced chef and answer the question which is related to cooking and recipes.'),
    HumanMessage(content='I dont like tomatoes, what else can make me a sandwich? Give a 2 line recipie.')
])

AIMessage(content='Skip the tomato worry!  Slather toasted bread with creamy avocado, then pile high with crispy bacon and crunchy sprouts. ', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-pro-001', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-6792458b-6ff9-4d27-bae0-141cc131ab2a-0', usage_metadata={'input_tokens': 41, 'output_tokens': 24, 'total_tokens': 65, 'input_token_details': {'cache_read': 0}})

In [13]:
from langchain import PromptTemplate

template = """
I want to be {career_option} in future. What subjects should I start studying?
Respond in 1-2 short sentence
"""

# Creating a template from the above prompt
prompt = PromptTemplate(
    input_variables=["career_option"],
    template=template
)

final_prompt = prompt.format(career_option='Machine Learning Engineer')

llm(final_prompt)

'Focus on building a strong foundation in mathematics (calculus, linear algebra, statistics, probability) and computer science (programming, algorithms, data structures). '

In [32]:
from langchain.prompts.example_selector import SemanticSimilarityExampleSelector
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.prompts import FewShotPromptTemplate
from langchain_community.vectorstores import FAISS
prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Example Input: {input}\nExample Output: {output}",
)
# Examples of job roles and respective job titles
examples = [
    {"input": "software engineer", "output": "software development"},
    {"input": "accountant", "output": "accounting"},
    {"input": "teacher", "output": "education"},
    {"input": "doctor", "output": "medicine"},
    {"input": "architect", "output": "architecture"},
    {"input": "lawyer", "output": "law"},
]
selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    GoogleGenerativeAIEmbeddings(model=specs['EMBED_MODEL_NAME'], google_api_key=specs['GOOGLE_API_KEY']),
    FAISS,
    k=2
)


In [33]:
similar_prompt = FewShotPromptTemplate(example_selector=selector, 
									   example_prompt=prompt,  
    								   prefix="Give the job title their job role is ", 
    								   suffix="Input: {job_title}\nOutput:",
    								   input_variables=["job_title"] 
    								   )

In [ ]:
print(similar_prompt.format(job_title='nurse'))


Give the job title their job role is 

Example Input: doctor
Example Output: medicine

Example Input: accountant
Example Output: accounting

Input: nurse
Output:


In [35]:
llm(similar_prompt.format(job_title='nurse'))

'Output: nursing '